> 在线讲义：[第 26 章 · 混合精度：bf16、fp16、TF32、FP8 怎么选](https://wikieden.github.io/pytorch-inside-out/lessons/26-mixed-precision.html) · 本书《PyTorch 从里到外》

# 第 26 章 · 混合精度：bf16、fp16、TF32、FP8 怎么选

本章回答：不同浮点格式的位是怎么分的，各自"能表示多大"和"能分辨多细"？为什么 bf16 不用 GradScaler 而 fp16 要？TF32 是什么，一行开关能快几倍？FP8 训练难在哪？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/26-mixed-precision.md`（讲解、动画、练习都在讲义里）。对应原文：Ultra-Scale Playbook §Mixed Precision Training（`ultrascale-playbook.md` 第 2821 行起；FP8 pretraining 第 2916 行）；多机报告 §4「精度：BF16 到 FP8」（`03-engineering-report.md` 第 286–340 行，含 DeepSeek-V3 FP8 配方）；官方文档 AMP 与 `torch.set_float32_matmul_precision`（〔在线〕）；本章 finfo 数字是本机 torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 finfo | max / eps / tiny | `finfo_table()` | fp16 max 65504、bf16 eps 0.0078、E4M3 max 448 |
| 第一步 舍入 | — | `rounding_examples()` | bf16 1.0、fp16 1.001、E4M3 3.25、fp16 溢出 inf |
| 第二步 loss scaling | 放大 65536 | `loss_scaling()` | 0.0 → 0.00066 |
| 第三步 TF32 | — | `get_float32_matmul_precision()` | highest（默认） |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `FORMATS`**（来自 `ch26_mixed_precision.py`）

In [ ]:
FORMATS = [torch.float32, torch.bfloat16, torch.float16, torch.float8_e4m3fn, torch.float8_e5m2]

**函数 `finfo_table`**（来自 `ch26_mixed_precision.py`）：下面要用到的小工具。

In [ ]:
def finfo_table():
    return {str(d).replace("torch.", ""): (torch.finfo(d).max, torch.finfo(d).eps, torch.finfo(d).tiny) for d in FORMATS}

**函数 `loss_scaling`**（来自 `ch26_mixed_precision.py`）：下面要用到的小工具。

In [ ]:
def loss_scaling(g: float = 1e-8, scale: float = 65536):
    return torch.tensor(g).half().item(), (torch.tensor(g) * scale).half().item()

**函数 `rounding_examples`**（来自 `ch26_mixed_precision.py`）：下面要用到的小工具。

In [ ]:
def rounding_examples():
    return dict(bf16=torch.tensor(1.001, dtype=torch.bfloat16).item(),
                fp16=round(torch.tensor(1.001, dtype=torch.float16).item(), 4),
                e4m3=torch.tensor(3.14159).to(torch.float8_e4m3fn).float().item(),
                fp16_overflow=(torch.tensor([1e4], dtype=torch.float16) * 10).item())

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 finfo (max, eps, tiny)：")
for k, v in finfo_table().items():
    print(f"  {k:14s} {v}")
print("舍入例子：", rounding_examples())

In [ ]:
print("\n第二步 梯度 1e-8 转 fp16 / 放大 65536 倍后转 fp16：", loss_scaling())

In [ ]:
print("\n第三步 fp32 矩阵乘默认精度：", torch.get_float32_matmul_precision())

## 3. 代码里才看得到的

1. **E4M3 的名字带 fn**：`float8_e4m3fn` 的 fn 表示没有 inf（finite），最大 448。

## 4. 自己改一改

- `torch.tensor(1.01, dtype=torch.bfloat16)` 看舍入成多少。
- 把 `loss_scaling` 的 scale 改成 1024，看还保不保得住 1e-8。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 指数管范围、尾数管精度；bf16 宽而粗，fp16 细而窄；
> fp16 梯度会下溢，GradScaler 先放大再除回；bf16 不用 scaler，累加记得用 fp32；
> fp32 一行开 TF32；FP8 再快一倍，靠缩放系数，粒度越细越稳。